# Torch2 Training Pipeline

Train the portfolio-ready V2 chess model and save artifacts with a dated name.

## Configuration and imports

Update `CONFIG` when you want to change dataset size, optimization settings, or the artifact prefix used during saving.

In [ ]:
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from chess import pgn
from torch.utils.data import DataLoader
from tqdm import tqdm

from auxiliary_func_v2 import (
    DEFAULT_MODEL_NAME,
    DEFAULT_MODELS_DIR,
    create_input_for_nn_v2,
    encode_moves,
    save_training_artifacts,
)
from dataset_v2 import ChessDatasetV2
from model_v2 import ChessModelV2

CONFIG = {
    'data_path': Path('../../data/png'),
    'max_games_to_load': 50000,
    'batch_size': 128,
    'epochs': 20,
    'learning_rate': 1e-4,
    'value_loss_weight': 0.5,
    'model_name': DEFAULT_MODEL_NAME,
    'models_dir': DEFAULT_MODELS_DIR,
}

CONFIG


## Load PGN games

The project currently stores PGN files in `data/png/` for historical reasons.

In [ ]:
def load_games(data_path: Path, max_games: int) -> list:
    games = []
    data_path = Path(data_path)
    pgn_files = sorted(data_path.glob('*.pgn'))

    if not pgn_files:
        raise FileNotFoundError(f'No .pgn files found in {data_path.resolve()}')

    for pgn_path in tqdm(pgn_files, desc='Loading PGN files'):
        if len(games) >= max_games:
            break

        with pgn_path.open('r', encoding='utf-8', errors='ignore') as pgn_file:
            while len(games) < max_games:
                game = pgn.read_game(pgn_file)
                if game is None:
                    break
                games.append(game)

    return games

print('Loading games...')
games = load_games(CONFIG['data_path'], CONFIG['max_games_to_load'])
print(f'Loaded {len(games)} games from {CONFIG["data_path"].resolve()}')


## Build tensors and labels

This step creates board tensors, move labels for the policy head, and outcome labels for the value head.

In [ ]:
print('Creating training data...')
X, y_policy, z_value = create_input_for_nn_v2(games)

print('Encoding moves...')
y_policy_encoded, move_to_int, _ = encode_moves(y_policy)
num_classes = len(move_to_int)

print(f'Samples: {len(X)} | Unique moves: {num_classes}')


## Train the V2 model

The model optimizes policy loss plus a weighted value loss.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

dataset = ChessDatasetV2(X, y_policy_encoded, z_value)
dataloader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0)

model = ChessModelV2(num_classes=num_classes).to(device)
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
policy_loss_fn = nn.CrossEntropyLoss()
value_loss_fn = nn.MSELoss()

print('Starting supervised training...')
for epoch in range(CONFIG['epochs']):
    model.train()
    total_loss = 0.0
    pbar = tqdm(dataloader, desc=f'Epoch {epoch + 1}/{CONFIG["epochs"]}')

    for board_states, policy_targets, value_targets in pbar:
        board_states = board_states.to(device)
        policy_targets = policy_targets.to(device)
        value_targets = value_targets.to(device).unsqueeze(1)

        optimizer.zero_grad()
        policy_logits, value_preds = model(board_states)

        policy_loss = policy_loss_fn(policy_logits, policy_targets)
        value_loss = value_loss_fn(value_preds, value_targets)
        loss = policy_loss + CONFIG['value_loss_weight'] * value_loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'policy': f'{policy_loss.item():.4f}',
            'value': f'{value_loss.item():.4f}',
        })

    avg_loss = total_loss / max(len(dataloader), 1)
    print(f'Epoch {epoch + 1} average loss: {avg_loss:.4f}')


## Save artifacts

The helper writes the checkpoint and move map together using the model name plus the training date.

In [ ]:
artifacts = save_training_artifacts(
    model=model,
    move_to_int=move_to_int,
    model_name=CONFIG['model_name'],
    models_dir=CONFIG['models_dir'],
)

print('Training complete.')
print(f"Model saved to: {artifacts['model_path']}")
print(f"Move map saved to: {artifacts['map_path']}")
print(f"Artifact prefix: {artifacts['artifact_prefix']}")
artifacts
